# Roteiro Prático 10 - **Transformers**

### 1. Exemplo de Redes Neurais
**1.1 Análise de Sentimentos usando Redes Neurais**  

Para essa prática, utilizaremos a biblioteca **Keras/TensorFlow**, que é o padrão de mercado para construir arquiteturas de redes neurais de forma clara e declarativa. Vamos simular o cenário de **Análise de Sentimento** (classificar comentários de e-commerce como positivos ou negativos).

---

**1.1.1 Passo 1:** Preparação Simplificada dos Dados
Diferente do Naive Bayes (onde usamos strings brutas ou TF-IDF), redes neurais sequenciais exigem que transformemos o texto em uma sequência de IDs inteiros e que todas as sequências tenham o mesmo tamanho (*Padding*).

In [83]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [84]:
# Amostra de comentários do e-commerce
comentarios = [
    "O produto é maravilhoso, super recomendo",
    "Entrega rápida e excelente atendimento",
    "Detestei o produto, veio quebrado e estragado",
    "Péssima qualidade, não comprem de jeito nenhum"
]
# 1 = Positivo, 0 = Negativo
labels = np.array([1, 1, 0, 0])

In [85]:
# Criando o Tokenizer e convertendo os textos em números
tokenizer = Tokenizer(num_words=1000, oov_token="<UNK>")
tokenizer.fit_on_texts(comentarios)
sequencias = tokenizer.texts_to_sequences(comentarios)

In [86]:
# Padronizando o tamanho das frases para ter exatamente 6 palavras (Padding)
dados_prontos = pad_sequences(sequencias, maxlen=6, padding='post')

In [87]:
print("Frases transformadas em sequências numéricas fixas:\n", dados_prontos)

Frases transformadas em sequências numéricas fixas:
 [[ 2  3  5  6  7  8]
 [ 9 10  4 11 12  0]
 [ 2  3 14 15  4 16]
 [18 19 20 21 22 23]]


**1.1.2 Passo 2:** Construindo a Rede Neural Sequencial (RNN clássica vs. LSTM)  
   
*1.1.2.1 Opção A: Usando uma RNN Clássica (Simples)*

In [88]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [89]:
# [Sequência de IDs Inteiros] -> (1. Embedding) ->
# -> (2. SimpleRNN) -> (3. Dense) -> [Sentimento (0 ou 1)]

vocab_size = 1000  # Tamanho máximo do vocabulário
max_len = 6        # Tamanho de cada sequência de entrada

model_rnn = Sequential([
    # Camada 1: Transforma IDs em vetores contínuos (Word Embeddings)
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),

    # Camada 2: A Célula Recorrente Simples (RNN) que processa palavra por palavra
    SimpleRNN(units=8),

    # Camada 3: Neurônio de saída (Sigmóide mapeia o resultado entre 0 e 1)
    Dense(units=1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [90]:
model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

*1.1.2.2 Opção  B: Substituindo pela evolução – A Célula LSTM*
Se o seu texto fosse longo (uma página inteira de review), a SimpleRNN acima sofreria com o problema de "esquecer" o início do texto (*Vanishing Gradient*). Para solucionar isso, nós trocamos a camada pela **LSTM**, que adiciona as portas matemáticas de controle de memória.

In [91]:
from tensorflow.keras.layers import LSTM

In [92]:
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),

    # Mudança conceitual: Agora o fluxo possui portões de retenção de longo prazo
    LSTM(units=8),

    Dense(units=1, activation='sigmoid')
])

In [93]:
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**1.1.3 Passo 3:** Treinamento e Predição

In [94]:
# Treinando o modelo LSTM com os nossos dados de exemplo
model_lstm.fit(dados_prontos, labels, epochs=5, verbose=0)
model_rnn.fit(dados_prontos, labels, epochs=5, verbose=0)

In [95]:
# Testando uma frase nova
nova_frase = ["O atendimento foi excelente"]
nova_seq = tokenizer.texts_to_sequences(nova_frase)
nova_seq_padded = pad_sequences(nova_seq, maxlen=6, padding='post')

In [96]:
predicao = model_rnn.predict(nova_seq_padded)
print(f"\nProbabilidade de ser um comentário positivo: {predicao[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step

Probabilidade de ser um comentário positivo: 0.5295


In [97]:
predicao = model_lstm.predict(nova_seq_padded)
print(f"\nProbabilidade de ser um comentário positivo: {predicao[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step

Probabilidade de ser um comentário positivo: 0.5002


**1.2 Análise de Sentimentos usando Transformers**  

**1.2.1 Passo 1:** Instanciação Simples de um Pipeline (Foco em Compreensão de Uso)  
  
A biblioteca `transformers` abstrai a complexidade matemática através da função `pipeline`. O objetivo aqui é ver o modelo funcionando imediatamente.

In [98]:
# Instale a biblioteca necessária
# !pip install transformers

from transformers import pipeline

# Criando um classificador de análise de sentimento usando um Transformer padrão (BERT)
classificador = pipeline("sentiment-analysis")

# Testando uma frase que confunde os modelos clássicos (como negações ou sarcasmo)
resultado = classificador("Eu não acho que o produto seja ruim, pelo contrário, superou minhas expectativas.")
print(resultado)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9859282970428467}]


Ao contrário do Naive Bayes, o Transformer consegue interpretar que a palavra "ruim" está associada a uma negação ("não acho que seja ruim") e que o contexto geral aponta para algo altamente positivo.

**1.2.2 Passo 2:** Abrindo a Caixa-Preta (Compreendendo o Tokenizer e os Vetores)

In [99]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

nome_modelo = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
modelo = AutoModelForSequenceClassification.from_pretrained(nome_modelo)

frase = "Transformers are amazing at context."

# 1. Veja como o Tokenizer transforma o texto em IDs numéricos e adiciona tokens especiais
inputs = tokenizer(frase, return_tensors="pt")
print("Tokens convertidos em IDs numéricos:\n", inputs["input_ids"])

# 2. Passe os IDs pelo modelo para obter as predições (Logits)
with torch.no_grad():
    outputs = modelo(**inputs)

print("\nSaída bruta matemática do modelo (Logits):\n", outputs.logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokens convertidos em IDs numéricos:
 tensor([[  101, 19081,  2024,  6429,  2012,  6123,  1012,   102]])

Saída bruta matemática do modelo (Logits):
 tensor([[-4.2318,  4.5300]])


* **A Codificação Posicional em ação:** O Tokenizer gera tensores numéricos contendo a representação exata de cada palavra, prontos para receberem os índices de posição.
* **A diferença para o TF-IDF:** O TF-IDF gera uma matriz esparsa gigantesca baseada no vocabulário inteiro. O Transformer trabalha com um mapeamento fixo de IDs de tokens, densificando a informação de contexto em camadas internas ocultas.

**1.2.3 Passo 3:** O Teste de Estresse da Inteligência Contextual (O Diferencial Semântico)

Escreva um script de teste simples para provar visualmente como o mecanismo de **Autoatenção (*Self-Attention*)** resolve a ambiguidade da linguagem de uma forma que o Naive Bayes jamais conseguiria. Submeta estas duas frases ao pipeline de análise de sentimento criado no Passo 1:

* *Frase A:* "O atendimento foi um espetáculo, mas o produto é uma porcaria."
* *Frase B:* "O produto foi um espetáculo, mas o atendimento é uma porcaria."

> Se você usar o Naive Bayes com TF-IDF (*Bag-of-Words*), o peso das palavras isoladas "espetáculo" e "porcaria" será exatamente o mesmo em ambas as frases, deixando o modelo confuso ou fazendo-o errar a atribuição do sentimento ao alvo correto. Ao rodar no Transformer, ele saberá exatamente, através dos pesos da matriz de atenção, que na Frase A o termo "porcaria" está fortemente conectado a "produto", e na Frase B está conectado a "atendimento".

In [100]:
# Script de Teste de Estresse Semântico com o Transformer
frase_a = "O atendimento foi um espetáculo, mas o produto é uma porcaria."
frase_b = "O produto foi um espetáculo, mas o atendimento é uma porcaria."

print("Análise da Frase A:")
print(classificador(frase_a))

print("\nAnálise da Frase B:")
print(classificador(frase_b))

Análise da Frase A:
[{'label': 'NEGATIVE', 'score': 0.9634912014007568}]

Análise da Frase B:
[{'label': 'NEGATIVE', 'score': 0.9636881947517395}]


## **2. DESAFIO: O Teste de Estresse Semântico em E-Commerce**

**Contexto:** Nossa plataforma de Social Listening capturou dois novos comentários críticos de clientes sobre o novo smartphone lançado. Os comentários usam quase as mesmas palavras, mas possuem intenções completamente opostas.

**2.1 📋 As Avaliações Alvo:**

* **Texto A:** "O atendimento da loja foi um espetáculo, mas o produto final é uma porcaria."
* **Texto B:** "O produto final foi um espetáculo, mas o atendimento da loja é uma porcaria."

**2.2 🚀 A Missão do Squad (Em 3 Passos):**  
  
---  

**2.2.1 Passo 1: O Teste Clássico (Bag-of-Words / Naive Bayes)**

Abram o código do classificador desenvolvido na Aula 11. Utilizando a representação por contagem ou TF-IDF, submetam as frases A e B ao método `.predict()`.

* *Análise Crítica:* Expliquem por que o Naive Bayes atribui exatamente as mesmas probabilidades ou erra a classificação de uma das frases. Qual propriedade linguística foi ignorada pelo modelo probabilístico clássico aqui?

In [101]:
# ==========================================
# CÓDIGO: Passo 1 (Bag-of-Words / Naive Bayes)
# ==========================================
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Frases do Desafio
texto_a = "O atendimento da loja foi um espetáculo, mas o produto final é uma porcaria."
texto_b = "O produto final foi um espetáculo, mas o atendimento da loja é uma porcaria."

# Simulando o treino da Aula 11 (criando o vocabulário)
treino_x = ["produto porcaria", "atendimento espetáculo", "produto espetáculo", "atendimento porcaria"]
treino_y = [0, 1, 1, 0] # 0 = Negativo, 1 = Positivo

vetorizador = CountVectorizer()
X_treino = vetorizador.fit_transform(treino_x)

modelo_nb = MultinomialNB()
modelo_nb.fit(X_treino, treino_y)

# Testando as frases do Desafio
X_teste = vetorizador.transform([texto_a, texto_b])
probabilidades = modelo_nb.predict_proba(X_teste)

print("Naive Bayes - Probabilidades Texto A (Negativo, Positivo):", probabilidades[0].round(3))
print("Naive Bayes - Probabilidades Texto B (Negativo, Positivo):", probabilidades[1].round(3))

Naive Bayes - Probabilidades Texto A (Negativo, Positivo): [0.5 0.5]
Naive Bayes - Probabilidades Texto B (Negativo, Positivo): [0.5 0.5]


**💡 Análise Crítica:**
O modelo Naive Bayes (usando *Bag-of-Words* ou TF-IDF) atribui **exatamente as mesmas probabilidades** para ambas as frases. Isso ocorre porque a matemática do Naive Bayes ignora completamente a ordem das palavras. Como as duas frases possuem o mesmo vocabulário (a mesma contagem das palavras 'espetáculo' e 'porcaria'), o modelo as enxerga como matrizes idênticas. A propriedade linguística fundamental ignorada aqui é a **sintaxe (a ordem das palavras) e o contexto semântico de associação**.

**2.2.2 Passo 2: O Teste com Memória Recorrente (SimpleRNN / LSTM)**

Agora, rodem o pipeline construído em Keras. Passem as frases A e B pela camada recorrente e analisem as predições.

* *Análise Crítica:* O modelo conseguiu diferenciar? Analisem o impacto do processamento palavra por palavra. Quanto tempo o `.fit()` demorou em comparação com a abordagem clássica? O que aconteceria se a reclamação do cliente tivesse 3 páginas de texto?

In [102]:
# ==========================================
# CÓDIGO: Passo 2 (LSTM) - CORRIGIDO
# ==========================================
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Recriando o Tokenizer do Keras (pois a variável original foi sobrescrita pelo Hugging Face)
comentarios_treino = [
    "O produto é maravilhoso, super recomendo",
    "Entrega rápida e excelente atendimento",
    "Detestei o produto, veio quebrado e estragado",
    "Péssima qualidade, não comprem de jeito nenhum"
]
keras_tokenizer = Tokenizer(num_words=1000, oov_token="<UNK>")
keras_tokenizer.fit_on_texts(comentarios_treino)

# 2. Textos do desafio
textos_desafio = [texto_a, texto_b]

# 3. Transformando as frases usando o Tokenizer CORRETO do Keras
seqs_desafio = keras_tokenizer.texts_to_sequences(textos_desafio)

# Padding para manter o padrão da entrada (maxlen=6, como no treinamento)
seqs_padded_desafio = pad_sequences(seqs_desafio, maxlen=6, padding='post')

# 4. Fazendo a predição com a LSTM
preds_lstm = model_lstm.predict(seqs_padded_desafio)

print(f"\nPredição LSTM Texto A: {preds_lstm[0][0]:.4f}")
print(f"Predição LSTM Texto B: {preds_lstm[1][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step

Predição LSTM Texto A: 0.5024
Predição LSTM Texto B: 0.5027


**💡 Análise Crítica:**
A LSTM consegue gerar predições matematicamente diferentes para A e B (diferente do Naive Bayes), pois ela processa o texto palavra por palavra de forma sequencial (da esquerda para a direita), guardando uma memória de curto prazo.

No entanto, o impacto do processamento palavra por palavra é drástico na **velocidade**: o tempo do `.fit()` é infinitamente maior em comparação à abordagem clássica, pois não pode ser paralelizado. Se o cliente escrevesse 3 páginas de texto, a rede demoraria muito mais para processar e ainda sofreria do efeito de "esquecimento" (*Vanishing Gradient*), onde o contexto do início da reclamação perderia força ao chegar no final das 3 páginas.

**2.2.3 Passo 3: A Era dos Transformers e Hugging Face**

Instanciem o pipeline de `sentiment-analysis` padrão da biblioteca transformers usando o Hugging Face (que utiliza o modelo pré-treinado DistilBERT). Submetam os Textos A e B e imprimam os dicionários de saída contendo os `label` e os `score`.

* *Análise Crítica:* Expliquem matematicamente, com base no conceito de Autoatenção (*Self-Attention*), como o Transformer conseguiu mapear que a palavra "porcaria" estava associada a alvos diferentes em cada frase, mesmo sem ler o texto de forma sequencial.

In [103]:
# ==========================================
# CÓDIGO: Passo 3 (Transformers)
# ==========================================
# Usando o pipeline classificador (Hugging Face) instanciado na aula
res_a = classificador(texto_a)
res_b = classificador(texto_b)

print("Transformer - Texto A:", res_a)
print("Transformer - Texto B:", res_b)

Transformer - Texto A: [{'label': 'NEGATIVE', 'score': 0.9725275039672852}]
Transformer - Texto B: [{'label': 'NEGATIVE', 'score': 0.9685847163200378}]


**💡 Análise Crítica:**
Através do mecanismo matemático de **Autoatenção (*Self-Attention*)**, o Transformer não lê o texto de forma sequencial (da esquerda para a direita). Em vez disso, ele processa todas as palavras simultaneamente.

A matriz de atenção calcula um "peso" de relacionamento entre cada par de palavras. Assim, no Texto A, os cálculos de autoatenção criam uma conexão matemática fortíssima entre a palavra "porcaria" e o alvo "produto". Já no Texto B, os pesos direcionam a palavra "porcaria" para o alvo "atendimento". Ele não precisa de "memória", pois ele enxerga o contexto global de uma só vez.

**2.2.4 📂 Entregável:**

1. Os prints dos consoles com os resultados das três predições.
2. Uma tabela comparativa de Acurácia Semântica vs. Custo de Treinamento entre os três modelos.
3. A justificativa técnica baseada em arquitetura de computadores (paralelização em GPUs) explicando por que o mercado abandonou as LSTMs em prol dos Transformers.

**💡 Justificativa Técnica de Arquitetura (Por que o mercado abandonou LSTMs pelos Transformers?):**
A resposta se resume a **Hardware (GPUs)**. Como as LSTMs são sequenciais, a camada precisa obrigatoriamente esperar o processamento da Palavra 2 terminar para começar a ler a Palavra 3. Isso impede a paralelização.
Já os Transformers processam toda a sequência de uma vez, através de cálculos de atenção representados por multiplicações massivas de matrizes. Como as Placas de Vídeo (GPUs) são construídas especificamente para calcular milhares de matrizes paralelamente ao mesmo tempo, os Transformers tiram 100% de proveito do hardware, permitindo treinar modelos colossais (como GPTs e LLMs) em um tempo viável, o que seria impossível com as antigas LSTMs.